# Thuc hanh 2 - Phan tich Log File

- Doc file `server.log` trong thu muc notebook
- Day du lieu len Spark cluster bang `parallelize`
- Loc dong `ERROR` va thong ke theo loai loi
- Co them cac phan mo rong: theo gio, canh bao, luu HDFS, top 5 IP


In [7]:
from pyspark.sql import SparkSession
import re

spark = (
    SparkSession.builder.appName('LogAnalysisNotebook')
    .master('spark://master:7077')
    .config('spark.pyspark.python', '/opt/conda/bin/python')
    .config('spark.cores.max', '4')
    .config('spark.executor.cores', '2')
    .getOrCreate()
)

sc = spark.sparkContext
print('Spark master =', sc.master)


Spark master = spark://master:7077


In [8]:
input_file = '/opt/workspace/notebooks/server.log'

with open(input_file, encoding='utf-8') as f:
    log_lines = [line.strip() for line in f if line.strip()]

print('So dong log doc duoc:', len(log_lines))
for line in log_lines:
    print(line)


So dong log doc duoc: 5
2024-01-15 10:23:01 ERROR DatabaseConnection timeout
2024-01-15 10:23:05 INFO User login successful
2024-01-15 10:23:10 ERROR FileNotFound /data/report.csv
2024-01-15 10:23:15 WARN Memory usage 85%
2024-01-15 10:23:20 ERROR DatabaseConnection refused


## Phan chinh

Loc cac dong `ERROR`, dem tong so loi, va thong ke theo loai loi.


In [9]:
# parallelize day du lieu tu driver len cluster de worker xu ly.
logs = sc.parallelize(log_lines, 4)
errors = logs.filter(lambda line: 'ERROR' in line)
total_errors = errors.count()

error_types = errors.map(lambda line: (line.split()[3], 1))
error_counts = error_types.reduceByKey(lambda a, b: a + b)
result = error_counts.sortBy(lambda item: (-item[1], item[0])).collect()

print(f'Tong so loi: {total_errors}')
print('LOG ANALYSIS RESULT')
for error_type, count in result:
    print(f'{error_type}\t{count}')


Tong so loi: 3
LOG ANALYSIS RESULT
DatabaseConnection	2
FileNotFound	1


## Mo rong 1 - Thong ke ERROR theo gio

Tu log mau hien tai, gio nam trong cot thu 2 o dang `HH:MM:SS`.


In [10]:
errors_by_hour = (
    errors
    .map(lambda line: (f"{line.split()[0]} {line.split()[1][:2]}", 1))
    .reduceByKey(lambda a, b: a + b)
    .sortBy(lambda item: item[0])
)

print('ERROR THEO GIO')
for hour_key, count in errors_by_hour.collect():
    print(f'{hour_key}\t{count}')


ERROR THEO GIO
2024-01-15 10	3


## Mo rong 2 - Canh bao khi so loi vuot nguong

Dat nguong tuy y. Neu tong so loi lon hon nguong thi in ra canh bao.


In [11]:
threshold = 2

if total_errors > threshold:
    print(f'ALERT: So loi = {total_errors}, vuot nguong {threshold}')
else:
    print(f'OK: So loi = {total_errors}, khong vuot nguong {threshold}')


ALERT: So loi = 3, vuot nguong 2


## Mo rong 3 - Luu ket qua ra HDFS

Cell nay xoa thu muc output cu neu da ton tai, sau do luu ket qua chinh va ket qua theo gio ra HDFS.


In [12]:
jvm = sc._jvm
hadoop_conf = sc._jsc.hadoopConfiguration()
fs = jvm.org.apache.hadoop.fs.FileSystem.get(jvm.java.net.URI('hdfs://master:9000'), hadoop_conf)

base_output = 'hdfs://master:9000/spark-events/log-analysis-notebook'
main_output = jvm.org.apache.hadoop.fs.Path(base_output + '/main')
hour_output = jvm.org.apache.hadoop.fs.Path(base_output + '/by-hour')

fs.delete(main_output, True)
fs.delete(hour_output, True)

error_counts.map(lambda item: f'{item[0]}\t{item[1]}').saveAsTextFile(base_output + '/main')
errors_by_hour.map(lambda item: f'{item[0]}\t{item[1]}').saveAsTextFile(base_output + '/by-hour')

print('Da luu ket qua vao:')
print(base_output + '/main')
print(base_output + '/by-hour')


Da luu ket qua vao:
hdfs://master:9000/spark-events/log-analysis-notebook/main
hdfs://master:9000/spark-events/log-analysis-notebook/by-hour


## Mo rong 4 - Tim 5 IP co nhieu loi nhat

Log mau hien tai khong co IP o dau dong. Cell nay se kiem tra schema truoc.
Neu token dau dong la IPv4 thi moi thong ke top 5 IP. Neu khong, no se bao khong co du lieu IP.


In [13]:
ip_regex = re.compile(r'^\d{1,3}(?:\.\d{1,3}){3}$')

error_ips = (
    errors
    .map(lambda line: line.split()[0])
    .filter(lambda token: ip_regex.match(token) is not None)
)

if error_ips.isEmpty():
    print('Khong tim thay truong IP trong log hien tai, nen khong the thong ke top 5 IP.')
else:
    top_ips = (
        error_ips
        .map(lambda ip: (ip, 1))
        .reduceByKey(lambda a, b: a + b)
        .takeOrdered(5, key=lambda item: -item[1])
    )
    print('TOP 5 IP NHIEU LOI NHAT')
    for ip, count in top_ips:
        print(f'{ip}\t{count}')


Khong tim thay truong IP trong log hien tai, nen khong the thong ke top 5 IP.
